In [1]:
import pandas as pd
import numpy as np
import re
import warnings
warnings.filterwarnings("ignore")

In [2]:
df = pd.read_csv("../data/raw/1429_1.csv")

In [3]:
# 1. Drop useless columns
df = df.drop(columns=[
    'reviews.didPurchase',
    'reviews.id',
    'reviews.userCity',
    'reviews.userProvince'
], errors='ignore')

In [4]:
# 2. Rename important columns
df = df.rename(columns={
    'name': 'product_name',
    'categories': 'category',
    'reviews.text': 'review_text',
    'reviews.rating': 'rating',
    'reviews.date': 'review_date',
    'reviews.title': 'review_title',
    'reviews.doRecommend': 'do_recommend'
})

In [5]:
# 3. Drop rows with missing rating or review text
df = df.dropna(subset=['review_text', 'rating'])

In [6]:
# 4. Convert rating to integer
df['rating'] = df['rating'].astype(int)

In [7]:
# 5. Generate sentiment
def rating_to_sentiment(x):
    if x >= 4:
        return "positive"
    elif x == 3:
        return "neutral"
    else:
        return "negative"

df['sentiment'] = df['rating'].apply(rating_to_sentiment)

In [8]:
# 6. Clean text
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+|https\S+", '', text)
    text = re.sub(r"<.*?>", '', text)
    text = re.sub(r"[^a-zA-Z0-9\s]", '', text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

df['clean_text'] = df['review_text'].apply(clean_text)

In [9]:
# 7. Convert dates
df['review_date'] = pd.to_datetime(df['review_date'], errors='coerce')

In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 34626 entries, 0 to 34659
Data columns (total 19 columns):
 #   Column              Non-Null Count  Dtype              
---  ------              --------------  -----              
 0   id                  34626 non-null  object             
 1   product_name        27867 non-null  object             
 2   asins               34624 non-null  object             
 3   brand               34626 non-null  object             
 4   category            34626 non-null  object             
 5   keys                34626 non-null  object             
 6   manufacturer        34626 non-null  object             
 7   review_date         34537 non-null  datetime64[ns, UTC]
 8   reviews.dateAdded   24039 non-null  object             
 9   reviews.dateSeen    34626 non-null  object             
 10  do_recommend        34066 non-null  object             
 11  reviews.numHelpful  34131 non-null  float64            
 12  rating              34626 non-null  i

In [11]:
df.head()

,id,product_name,asins,brand,category,keys,manufacturer,review_date,reviews.dateAdded,reviews.dateSeen,do_recommend,reviews.numHelpful,rating,reviews.sourceURLs,review_text,review_title,reviews.username,sentiment,clean_text
0,AVqkIhwDv8e3D1O-lebb,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",B01AHB9CN2,Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Ta...","841667104676,amazon/53004484,amazon/b01ahb9cn2...",Amazon,2017-01-13 00:00:00+00:00,2017-07-03T23:33:15Z,"2017-06-07T09:04:00.000Z,2017-04-30T00:45:00.000Z",True,0.0,5,http://reviews.bestbuy.com/3545/5620406/review...,This product so far has not disappointed. My c...,Kindle,Adapter,positive,this product so far has not disappointed my ch...
1,AVqkIhwDv8e3D1O-lebb,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",B01AHB9CN2,Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Ta...","841667104676,amazon/53004484,amazon/b01ahb9cn2...",Amazon,2017-01-13 00:00:00+00:00,2017-07-03T23:33:15Z,"2017-06-07T09:04:00.000Z,2017-04-30T00:45:00.000Z",True,0.0,5,http://reviews.bestbuy.com/3545/5620406/review...,great for beginner or experienced person. Boug...,very fast,truman,positive,great for beginner or experienced person bough...
2,AVqkIhwDv8e3D1O-lebb,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",B01AHB9CN2,Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Ta...","841667104676,amazon/53004484,amazon/b01ahb9cn2...",Amazon,2017-01-13 00:00:00+00:00,2017-07-03T23:33:15Z,"2017-06-07T09:04:00.000Z,2017-04-30T00:45:00.000Z",True,0.0,5,http://reviews.bestbuy.com/3545/5620406/review...,Inexpensive tablet for him to use and learn on...,Beginner tablet for our 9 year old son.,DaveZ,positive,inexpensive tablet for him to use and learn on...
3,AVqkIhwDv8e3D1O-lebb,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",B01AHB9CN2,Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Ta...","841667104676,amazon/53004484,amazon/b01ahb9cn2...",Amazon,2017-01-13 00:00:00+00:00,2017-07-03T23:33:15Z,"2017-06-07T09:04:00.000Z,2017-04-30T00:45:00.000Z",True,0.0,4,http://reviews.bestbuy.com/3545/5620406/review...,I've had my Fire HD 8 two weeks now and I love...,Good!!!,Shacks,positive,ive had my fire hd 8 two weeks now and i love ...
4,AVqkIhwDv8e3D1O-lebb,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",B01AHB9CN2,Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Ta...","841667104676,amazon/53004484,amazon/b01ahb9cn2...",Amazon,2017-01-12 00:00:00+00:00,2017-07-03T23:33:15Z,"2017-06-07T09:04:00.000Z,2017-04-30T00:45:00.000Z",True,0.0,5,http://reviews.bestbuy.com/3545/5620406/review...,I bought this for my grand daughter when she c...,Fantastic Tablet for kids,explore42,positive,i bought this for my grand daughter when she c...


In [13]:
df2 = pd.read_csv("../data/raw/Datafiniti_Amazon_Consumer_Reviews_of_Amazon_Products.csv")
df3 = pd.read_csv("../data/raw/Datafiniti_Amazon_Consumer_Reviews_of_Amazon_Products_May19.csv")


In [15]:
df2.shape

(5000, 24)

In [16]:
df2.head()

,id,dateAdded,dateUpdated,name,asins,brand,categories,primaryCategories,imageURLs,keys,...,reviews.dateSeen,reviews.doRecommend,reviews.id,reviews.numHelpful,reviews.rating,reviews.sourceURLs,reviews.text,reviews.title,reviews.username,sourceURLs
0,AVqVGZNvQMlgsOJE6eUY,2017-03-03T16:56:05Z,2018-10-25T16:36:31Z,"Amazon Kindle E-Reader 6"" Wifi (8th Generation...",B00ZV9PXP2,Amazon,"Computers,Electronics Features,Tablets,Electro...",Electronics,https://pisces.bbystatic.com/image2/BestBuy_US...,allnewkindleereaderblack6glarefreetouchscreend...,...,"2018-05-27T00:00:00Z,2017-09-18T00:00:00Z,2017...",False,NaN,0,3,http://reviews.bestbuy.com/3545/5442403/review...,I thought it would be as big as small paper bu...,Too small,llyyue,https://www.newegg.com/Product/Product.aspx%25...
1,AVqVGZNvQMlgsOJE6eUY,2017-03-03T16:56:05Z,2018-10-25T16:36:31Z,"Amazon Kindle E-Reader 6"" Wifi (8th Generation...",B00ZV9PXP2,Amazon,"Computers,Electronics Features,Tablets,Electro...",Electronics,https://pisces.bbystatic.com/image2/BestBuy_US...,allnewkindleereaderblack6glarefreetouchscreend...,...,"2018-05-27T00:00:00Z,2017-07-07T00:00:00Z,2017...",True,NaN,0,5,http://reviews.bestbuy.com/3545/5442403/review...,This kindle is light and easy to use especiall...,Great light reader. Easy to use at the beach,Charmi,https://www.newegg.com/Product/Product.aspx%25...
2,AVqVGZNvQMlgsOJE6eUY,2017-03-03T16:56:05Z,2018-10-25T16:36:31Z,"Amazon Kindle E-Reader 6"" Wifi (8th Generation...",B00ZV9PXP2,Amazon,"Computers,Electronics Features,Tablets,Electro...",Electronics,https://pisces.bbystatic.com/image2/BestBuy_US...,allnewkindleereaderblack6glarefreetouchscreend...,...,2018-05-27T00:00:00Z,True,NaN,0,4,https://reviews.bestbuy.com/3545/5442403/revie...,Didnt know how much i'd use a kindle so went f...,Great for the price,johnnyjojojo,https://www.newegg.com/Product/Product.aspx%25...
3,AVqVGZNvQMlgsOJE6eUY,2017-03-03T16:56:05Z,2018-10-25T16:36:31Z,"Amazon Kindle E-Reader 6"" Wifi (8th Generation...",B00ZV9PXP2,Amazon,"Computers,Electronics Features,Tablets,Electro...",Electronics,https://pisces.bbystatic.com/image2/BestBuy_US...,allnewkindleereaderblack6glarefreetouchscreend...,...,2018-10-09T00:00:00Z,True,177283626.0,3,5,https://redsky.target.com/groot-domain-api/v1/...,I am 100 happy with my purchase. I caught it o...,A Great Buy,Kdperry,https://www.newegg.com/Product/Product.aspx%25...
4,AVqVGZNvQMlgsOJE6eUY,2017-03-03T16:56:05Z,2018-10-25T16:36:31Z,"Amazon Kindle E-Reader 6"" Wifi (8th Generation...",B00ZV9PXP2,Amazon,"Computers,Electronics Features,Tablets,Electro...",Electronics,https://pisces.bbystatic.com/image2/BestBuy_US...,allnewkindleereaderblack6glarefreetouchscreend...,...,2018-05-27T00:00:00Z,True,NaN,0,5,https://reviews.bestbuy.com/3545/5442403/revie...,Solid entry level Kindle. Great for kids. Gift...,Solid entry-level Kindle. Great for kids,Johnnyblack,https://www.newegg.com/Product/Product.aspx%25...


In [17]:
df3.shape

(28332, 24)

In [18]:
df3.head()

,id,dateAdded,dateUpdated,name,asins,brand,categories,primaryCategories,imageURLs,keys,...,reviews.didPurchase,reviews.doRecommend,reviews.id,reviews.numHelpful,reviews.rating,reviews.sourceURLs,reviews.text,reviews.title,reviews.username,sourceURLs
0,AVpgNzjwLJeJML43Kpxn,2015-10-30T08:59:32Z,2019-04-25T09:08:16Z,AmazonBasics AAA Performance Alkaline Batterie...,"B00QWO9P0O,B00LH3DMUO",Amazonbasics,"AA,AAA,Health,Electronics,Health & Household,C...",Health & Beauty,https://images-na.ssl-images-amazon.com/images...,"amazonbasics/hl002619,amazonbasicsaaaperforman...",...,NaN,NaN,NaN,NaN,3,https://www.amazon.com/product-reviews/B00QWO9...,I order 3 of them and one of the item is bad q...,... 3 of them and one of the item is bad quali...,Byger yang,"https://www.barcodable.com/upc/841710106442,ht..."
1,AVpgNzjwLJeJML43Kpxn,2015-10-30T08:59:32Z,2019-04-25T09:08:16Z,AmazonBasics AAA Performance Alkaline Batterie...,"B00QWO9P0O,B00LH3DMUO",Amazonbasics,"AA,AAA,Health,Electronics,Health & Household,C...",Health & Beauty,https://images-na.ssl-images-amazon.com/images...,"amazonbasics/hl002619,amazonbasicsaaaperforman...",...,NaN,NaN,NaN,NaN,4,https://www.amazon.com/product-reviews/B00QWO9...,Bulk is always the less expensive way to go fo...,... always the less expensive way to go for pr...,ByMG,"https://www.barcodable.com/upc/841710106442,ht..."
2,AVpgNzjwLJeJML43Kpxn,2015-10-30T08:59:32Z,2019-04-25T09:08:16Z,AmazonBasics AAA Performance Alkaline Batterie...,"B00QWO9P0O,B00LH3DMUO",Amazonbasics,"AA,AAA,Health,Electronics,Health & Household,C...",Health & Beauty,https://images-na.ssl-images-amazon.com/images...,"amazonbasics/hl002619,amazonbasicsaaaperforman...",...,NaN,NaN,NaN,NaN,5,https://www.amazon.com/product-reviews/B00QWO9...,Well they are not Duracell but for the price i...,... are not Duracell but for the price i am ha...,BySharon Lambert,"https://www.barcodable.com/upc/841710106442,ht..."
3,AVpgNzjwLJeJML43Kpxn,2015-10-30T08:59:32Z,2019-04-25T09:08:16Z,AmazonBasics AAA Performance Alkaline Batterie...,"B00QWO9P0O,B00LH3DMUO",Amazonbasics,"AA,AAA,Health,Electronics,Health & Household,C...",Health & Beauty,https://images-na.ssl-images-amazon.com/images...,"amazonbasics/hl002619,amazonbasicsaaaperforman...",...,NaN,NaN,NaN,NaN,5,https://www.amazon.com/product-reviews/B00QWO9...,Seem to work as well as name brand batteries a...,... as well as name brand batteries at a much ...,Bymark sexson,"https://www.barcodable.com/upc/841710106442,ht..."
4,AVpgNzjwLJeJML43Kpxn,2015-10-30T08:59:32Z,2019-04-25T09:08:16Z,AmazonBasics AAA Performance Alkaline Batterie...,"B00QWO9P0O,B00LH3DMUO",Amazonbasics,"AA,AAA,Health,Electronics,Health & Household,C...",Health & Beauty,https://images-na.ssl-images-amazon.com/images...,"amazonbasics/hl002619,amazonbasicsaaaperforman...",...,NaN,NaN,NaN,NaN,5,https://www.amazon.com/product-reviews/B00QWO9...,These batteries are very long lasting the pric...,... batteries are very long lasting the price ...,Bylinda,"https://www.barcodable.com/upc/841710106442,ht..."


In [19]:
rename_map = {
    'name': 'product_name',
    'categories': 'category',
    'reviews.text': 'review_text',
    'reviews.title': 'review_title',
    'reviews.date': 'review_date',
    'reviews.doRecommend': 'do_recommend',
    'reviews.rating': 'rating',
}
df2.rename(columns=rename_map, inplace=True)
df3.rename(columns=rename_map, inplace=True)

In [20]:
df2['reviews.didPurchase'] = df2.get('reviews.didPurchase', None)
df3['reviews.dateAdded'] = df3.get('reviews.dateAdded', None)

In [21]:
df2['review_date'] = pd.to_datetime(df2['review_date'], errors='coerce', utc=True)
df3['review_date'] = pd.to_datetime(df3['review_date'], errors='coerce', utc=True)

In [22]:
df2['clean_text'] = df2['review_text'].astype(str).apply(clean_text)
df3['clean_text'] = df3['review_text'].astype(str).apply(clean_text)

In [24]:
df2['sentiment'] = df2['rating'].apply(rating_to_sentiment)
df3['sentiment'] = df3['rating'].apply(rating_to_sentiment)

In [26]:
combined_df = pd.concat([df, df2, df3], ignore_index=True)

In [27]:
combined_df.shape

(67958, 27)

In [28]:
combined_df.head()

,id,product_name,asins,brand,category,keys,manufacturer,review_date,reviews.dateAdded,reviews.dateSeen,...,sentiment,clean_text,dateAdded,dateUpdated,primaryCategories,imageURLs,manufacturerNumber,reviews.id,sourceURLs,reviews.didPurchase
0,AVqkIhwDv8e3D1O-lebb,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",B01AHB9CN2,Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Ta...","841667104676,amazon/53004484,amazon/b01ahb9cn2...",Amazon,2017-01-13 00:00:00+00:00,2017-07-03T23:33:15Z,"2017-06-07T09:04:00.000Z,2017-04-30T00:45:00.000Z",...,positive,this product so far has not disappointed my ch...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,AVqkIhwDv8e3D1O-lebb,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",B01AHB9CN2,Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Ta...","841667104676,amazon/53004484,amazon/b01ahb9cn2...",Amazon,2017-01-13 00:00:00+00:00,2017-07-03T23:33:15Z,"2017-06-07T09:04:00.000Z,2017-04-30T00:45:00.000Z",...,positive,great for beginner or experienced person bough...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,AVqkIhwDv8e3D1O-lebb,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",B01AHB9CN2,Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Ta...","841667104676,amazon/53004484,amazon/b01ahb9cn2...",Amazon,2017-01-13 00:00:00+00:00,2017-07-03T23:33:15Z,"2017-06-07T09:04:00.000Z,2017-04-30T00:45:00.000Z",...,positive,inexpensive tablet for him to use and learn on...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,AVqkIhwDv8e3D1O-lebb,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",B01AHB9CN2,Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Ta...","841667104676,amazon/53004484,amazon/b01ahb9cn2...",Amazon,2017-01-13 00:00:00+00:00,2017-07-03T23:33:15Z,"2017-06-07T09:04:00.000Z,2017-04-30T00:45:00.000Z",...,positive,ive had my fire hd 8 two weeks now and i love ...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,AVqkIhwDv8e3D1O-lebb,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",B01AHB9CN2,Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Ta...","841667104676,amazon/53004484,amazon/b01ahb9cn2...",Amazon,2017-01-12 00:00:00+00:00,2017-07-03T23:33:15Z,"2017-06-07T09:04:00.000Z,2017-04-30T00:45:00.000Z",...,positive,i bought this for my grand daughter when she c...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [30]:
combined_df.info()
combined_df.head(3)
combined_df.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 67958 entries, 0 to 67957
Data columns (total 27 columns):
 #   Column               Non-Null Count  Dtype              
---  ------               --------------  -----              
 0   id                   67958 non-null  object             
 1   product_name         61199 non-null  object             
 2   asins                67956 non-null  object             
 3   brand                67958 non-null  object             
 4   category             67958 non-null  object             
 5   keys                 67958 non-null  object             
 6   manufacturer         67958 non-null  object             
 7   review_date          67807 non-null  datetime64[ns, UTC]
 8   reviews.dateAdded    25091 non-null  object             
 9   reviews.dateSeen     67958 non-null  object             
 10  do_recommend         55152 non-null  object             
 11  reviews.numHelpful   55246 non-null  float64            
 12  rating            

id                         0
product_name            6759
asins                      2
brand                      0
category                   0
keys                       0
manufacturer               0
review_date              151
reviews.dateAdded      42867
reviews.dateSeen           0
do_recommend           12806
reviews.numHelpful     12712
rating                     0
reviews.sourceURLs         0
review_text                0
review_title              19
reviews.username          13
sentiment                  0
clean_text                 0
dateAdded              34626
dateUpdated            34626
primaryCategories      34626
imageURLs              34626
manufacturerNumber     34626
reviews.id             67888
sourceURLs             34626
reviews.didPurchase    67949
dtype: int64

In [31]:
cols_to_drop = [
    'reviews.id',
    'reviews.didPurchase',
    'dateAdded',
    'dateUpdated',
    'primaryCategories',
    'imageURLs',
    'manufacturerNumber',
    'sourceURLs',           # duplicate
    'reviews.dateAdded',    # mostly null, inconsistent
]

combined_df.drop(columns=cols_to_drop, inplace=True, errors='ignore')

In [32]:
combined_df['do_recommend'] = combined_df['do_recommend'].map({
    True: 1, 
    False: 0,
    'True': 1,
    'False': 0
})

In [33]:
combined_df = combined_df.dropna(subset=['review_date'])

In [34]:
combined_df.reset_index(drop=True, inplace=True)

In [36]:
combined_df.info()
combined_df.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 67807 entries, 0 to 67806
Data columns (total 18 columns):
 #   Column              Non-Null Count  Dtype              
---  ------              --------------  -----              
 0   id                  67807 non-null  object             
 1   product_name        61112 non-null  object             
 2   asins               67807 non-null  object             
 3   brand               67807 non-null  object             
 4   category            67807 non-null  object             
 5   keys                67807 non-null  object             
 6   manufacturer        67807 non-null  object             
 7   review_date         67807 non-null  datetime64[ns, UTC]
 8   reviews.dateSeen    67807 non-null  object             
 9   do_recommend        55152 non-null  float64            
 10  reviews.numHelpful  55214 non-null  float64            
 11  rating              67807 non-null  int64              
 12  reviews.sourceURLs  67807 non-nu

id                        0
product_name           6695
asins                     0
brand                     0
category                  0
keys                      0
manufacturer              0
review_date               0
reviews.dateSeen          0
do_recommend          12655
reviews.numHelpful    12593
rating                    0
reviews.sourceURLs        0
review_text               0
review_title             19
reviews.username         13
sentiment                 0
clean_text                0
dtype: int64

In [38]:
combined_df.to_csv("../data/processed/amazon_reviews_cleaned.csv", index=False)